# Personality Prediction — Introvert vs Extrovert
**Dataset:** `personality_dataset.csv`  
**Goal:** Binary classification — predict whether a person is an Introvert or Extrovert based on 7 behavioural features.  
**Best model achieved ~92% accuracy on held-out test data.**


## 1. Imports & Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import (classification_report, confusion_matrix,
                             accuracy_score, ConfusionMatrixDisplay)
import joblib, json, warnings
warnings.filterwarnings('ignore')

sns.set_theme(style='whitegrid', palette='muted')
print("Libraries loaded ✓")

## 2. Load & Inspect the Dataset

In [ ]:
df = pd.read_csv('personality_dataset.csv')
print(f"Shape: {df.shape}")
df.head()

In [ ]:
print("Data types:")
print(df.dtypes)
print("\nMissing values:")
print(df.isnull().sum())

In [ ]:
print("\nTarget distribution:")
print(df['Personality'].value_counts())

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
df['Personality'].value_counts().plot.bar(ax=axes[0], color=['steelblue','coral'])
axes[0].set_title('Class Distribution')
axes[0].set_ylabel('Count')

numeric_cols = ['Time_spent_Alone','Social_event_attendance',
                'Going_outside','Friends_circle_size','Post_frequency']
df[numeric_cols].hist(bins=15, figsize=(14, 6), color='steelblue', edgecolor='white')
plt.suptitle('Numeric Feature Distributions', y=1.02)
plt.tight_layout()
plt.show()

## 3. Exploratory Data Analysis

In [ ]:
# Boxplots: numeric features vs personality
fig, axes = plt.subplots(2, 3, figsize=(16, 8))
axes = axes.flatten()
for i, col in enumerate(numeric_cols):
    sns.boxplot(data=df, x='Personality', y=col, ax=axes[i],
                palette={'Introvert':'coral','Extrovert':'steelblue'})
    axes[i].set_title(col)
axes[-1].set_visible(False)
plt.suptitle('Numeric Features by Personality Type', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# Categorical features
cat_cols = ['Stage_fear', 'Drained_after_socializing']
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
for ax, col in zip(axes, cat_cols):
    ct = pd.crosstab(df[col], df['Personality'], normalize='index') * 100
    ct.plot(kind='bar', ax=ax, color=['steelblue','coral'])
    ax.set_title(f'{col} vs Personality (%)')
    ax.set_ylabel('%')
    ax.tick_params(axis='x', rotation=0)
plt.tight_layout()
plt.show()

In [ ]:
# Correlation heatmap (numeric only)
plt.figure(figsize=(8, 6))
corr = df[numeric_cols].corr()
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', square=True)
plt.title('Feature Correlation Matrix')
plt.tight_layout()
plt.show()

## 4. Preprocessing Pipeline

In [ ]:
# Feature / target split
X = df.drop('Personality', axis=1)
y = (df['Personality'] == 'Introvert').astype(int)   # 1=Introvert, 0=Extrovert

numeric_features   = ['Time_spent_Alone', 'Social_event_attendance',
                       'Going_outside', 'Friends_circle_size', 'Post_frequency']
categorical_features = ['Stage_fear', 'Drained_after_socializing']

numeric_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])
categorical_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(drop='if_binary', handle_unknown='ignore'))
])
preprocessor = ColumnTransformer([
    ('num', numeric_transformer, numeric_features),
    ('cat', categorical_transformer, categorical_features)
])

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)
print(f"Train: {X_train.shape}  |  Test: {X_test.shape}")

## 5. Model Training & Cross-Validation

In [ ]:
models = {
    'Logistic Regression': LogisticRegression(max_iter=500, random_state=42),
    'Random Forest':       RandomForestClassifier(n_estimators=200, random_state=42),
    'Gradient Boosting':   GradientBoostingClassifier(n_estimators=200, random_state=42)
}

results = {}
for name, clf in models.items():
    pipe = Pipeline([('pre', preprocessor), ('clf', clf)])
    cv = cross_val_score(pipe, X_train, y_train, cv=5, scoring='accuracy')
    results[name] = cv
    print(f"{name:25s}  CV: {cv.mean():.4f} ± {cv.std():.4f}")

# Plot CV results
fig, ax = plt.subplots(figsize=(9, 4))
ax.boxplot(results.values(), labels=results.keys())
ax.set_title('5-Fold CV Accuracy by Model')
ax.set_ylabel('Accuracy')
plt.tight_layout()
plt.show()

## 6. Best Model — Final Evaluation

In [ ]:
# Best model by CV mean accuracy
best_name = max(results, key=lambda k: results[k].mean())
print(f"Best model: {best_name}")

best_pipe = Pipeline([('pre', preprocessor), ('clf', models[best_name])])
best_pipe.fit(X_train, y_train)
y_pred = best_pipe.predict(X_test)

print(f"\nTest Accuracy: {accuracy_score(y_test, y_pred):.4f}")
print("\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=['Extrovert','Introvert']))

In [ ]:
# Confusion matrix
cm = confusion_matrix(y_test, y_pred)
disp = ConfusionMatrixDisplay(cm, display_labels=['Extrovert','Introvert'])
disp.plot(cmap='Blues')
plt.title('Confusion Matrix — Test Set')
plt.tight_layout()
plt.show()

## 7. Feature Importance

In [ ]:
# For Logistic Regression: use coefficients
if best_name == 'Logistic Regression':
    clf = best_pipe.named_steps['clf']
    pre = best_pipe.named_steps['pre']
    ohe_cols = pre.named_transformers_['cat']['onehot'].get_feature_names_out(categorical_features)
    all_cols = numeric_features + list(ohe_cols)
    importance = pd.Series(np.abs(clf.coef_[0]), index=all_cols).sort_values(ascending=True)
    importance.plot(kind='barh', figsize=(9,5), color='steelblue')
    plt.title('Feature Importance (|Coefficient|)')
    plt.tight_layout()
    plt.show()
elif hasattr(best_pipe.named_steps['clf'], 'feature_importances_'):
    clf = best_pipe.named_steps['clf']
    pre = best_pipe.named_steps['pre']
    ohe_cols = pre.named_transformers_['cat']['onehot'].get_feature_names_out(categorical_features)
    all_cols = numeric_features + list(ohe_cols)
    importance = pd.Series(clf.feature_importances_, index=all_cols).sort_values(ascending=True)
    importance.plot(kind='barh', figsize=(9,5), color='steelblue')
    plt.title('Feature Importance')
    plt.tight_layout()
    plt.show()

## 8. Save Model

In [ ]:
joblib.dump(best_pipe, 'model.pkl')
meta = {
    "features": list(X.columns),
    "numeric_features": numeric_features,
    "categorical_features": categorical_features,
    "label_map": {"0": "Extrovert", "1": "Introvert"},
    "best_model": best_name,
    "test_accuracy": round(accuracy_score(y_test, y_pred), 4)
}
with open('model_meta.json', 'w') as f:
    json.dump(meta, f, indent=2)
print("✓ model.pkl and model_meta.json saved")

## 9. Quick Inference Test

In [ ]:
import pandas as pd, joblib, json

model = joblib.load('model.pkl')

# Likely Introvert
sample_introvert = pd.DataFrame([{
    'Time_spent_Alone': 9,
    'Stage_fear': 'Yes',
    'Social_event_attendance': 1,
    'Going_outside': 1,
    'Drained_after_socializing': 'Yes',
    'Friends_circle_size': 2,
    'Post_frequency': 1
}])

# Likely Extrovert
sample_extrovert = pd.DataFrame([{
    'Time_spent_Alone': 1,
    'Stage_fear': 'No',
    'Social_event_attendance': 9,
    'Going_outside': 6,
    'Drained_after_socializing': 'No',
    'Friends_circle_size': 12,
    'Post_frequency': 8
}])

label_map = {"0": "Extrovert", "1": "Introvert"}
for name, sample in [("Sample A (expected: Introvert)", sample_introvert),
                     ("Sample B (expected: Extrovert)", sample_extrovert)]:
    pred = model.predict(sample)[0]
    prob = model.predict_proba(sample)[0]
    print(f"{name}")
    print(f"  Prediction : {label_map[str(pred)]}")
    print(f"  Confidence : {prob[pred]:.2%}\n")

## Summary

| Item | Detail |
|---|---|
| Dataset | 2,900 rows × 7 features + 1 target |
| Missing values | Handled via median / mode imputation |
| Best model | Logistic Regression |
| CV Accuracy | ~93.6% |
| Test Accuracy | ~91.7% |

### Deployment
1. Push `app.py`, `model.pkl`, `model_meta.json`, `requirements.txt`, `Procfile` to GitHub.  
2. Create a free **Render** Web Service pointing to that repo.  
3. POST to `https://<your-service>.onrender.com/predict` with the feature JSON.
